# Thermodynamic Pruning of Neural Networks: Ising-Inspired Sparsity for Efficient Alzheimer's Classification

**Dataset:** [Augmented Alzheimer MRI Dataset](https://www.kaggle.com/datasets/amananandrai/augmented-alzheimer-mri-dataset)

In [ ]:
import numpy as np
import pandas as pd
import os
import random
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from copy import deepcopy

# ---------------- Reproducibility ----------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ---------------- Config ----------------
BASE_PATH = "/kaggle/input/datasets/uraninjo/augmented-alzheimer-mri-dataset/AugmentedAlzheimerDataset"
CATEGORIES = ["MildDemented", "ModerateDemented", "NonDemented", "VeryMildDemented"]
CLASS_NAMES = ["Mild", "Mod", "Non", "VeryMild"]
IMG_SIZE = 128
BATCH_SIZE = 64
EPOCHS = 20
FINETUNE_EPOCHS = 20
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 1. Build the dataframe

In [ ]:
image_paths, labels = [], []
for category in CATEGORIES:
    category_path = os.path.join(BASE_PATH, category)
    for image_name in os.listdir(category_path):
        image_paths.append(os.path.join(category_path, image_name))
        labels.append(category)

df = pd.DataFrame({"image_path": image_paths, "label": labels})
print(df.shape)
print(df["label"].value_counts())
print("Nulls:", df.isnull().sum().sum(), "| Duplicates:", df.duplicated().sum())

## 2. EDA

In [ ]:
sns.set_theme(style="whitegrid", context="talk", palette="viridis")
fig, ax = plt.subplots(figsize=(9, 6))
sns.countplot(data=df, x="label", ax=ax)
ax.set_title("Class Distribution", fontsize=16, fontweight="bold", pad=15)
ax.set_xlabel("Class"); ax.set_ylabel("Count")
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
for p in ax.patches:
    ax.annotate(f"{p.get_height():,}", (p.get_x()+p.get_width()/2., p.get_height()),
                ha="center", va="bottom", fontsize=11, xytext=(0, 6), textcoords="offset points")
sns.despine(); plt.tight_layout(); plt.show()

# Sample images per class
import cv2
num_images = 5
plt.figure(figsize=(15, 12))
for i, category in enumerate(CATEGORIES):
    sample_paths = df[df["label"] == category]["image_path"].head(num_images).values
    for j, img_path in enumerate(sample_paths):
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        ax = plt.subplot(len(CATEGORIES), num_images, i*num_images + j + 1)
        ax.imshow(img); ax.set_title(category, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## 3. Leakage-free split FIRST, oversample train ONLY

In [ ]:
# Stratified train/val/test on the ORIGINAL dataframe (no augmentation leakage)
train_df, temp_df = train_test_split(df, test_size=0.20, stratify=df["label"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED)

# Oversample ONLY the training set to balance classes
max_samples = train_df["label"].value_counts().max()
train_df = (train_df.groupby("label", group_keys=False)
            .apply(lambda x: x.sample(n=max_samples, replace=True, random_state=SEED))
            .reset_index(drop=True))
train_df = train_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)  # shuffle

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(train_df["label"].value_counts())

## 4. Dataset & DataLoaders

In [ ]:
class MRIDataset(Dataset):
    def __init__(self, df, transform=None):
        self.paths = df["image_path"].values
        self.labels = pd.Categorical(df["label"], categories=CATEGORIES).codes.astype(np.int64)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_loader = DataLoader(MRIDataset(train_df, transform), batch_size=BATCH_SIZE,
                          shuffle=True, pin_memory=True, num_workers=2)
val_loader   = DataLoader(MRIDataset(val_df, transform),   batch_size=512,
                          shuffle=False, pin_memory=True, num_workers=2)
test_loader  = DataLoader(MRIDataset(test_df, transform),  batch_size=512,
                          shuffle=False, pin_memory=True, num_workers=2)

## 5. Model MLP with proper (Kaiming) initialization


In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(IMG_SIZE*IMG_SIZE*3, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, len(CATEGORIES)),
        )
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, nonlinearity="relu")
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)

model = SimpleMLP().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## 6. Training loop (tracks REAL train accuracy)

In [ ]:
def train_epoch(model, optimizer):
    model.train()
    correct, total, running_loss = 0, 0, 0.0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        loss = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)  # stability
        optimizer.step()
        running_loss += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)
    return correct / total, running_loss / total

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, trues = [], []
    for x, y in loader:
        x = x.to(DEVICE)
        preds.extend(model(x).argmax(1).cpu().numpy())
        trues.extend(y.numpy())
    preds, trues = np.array(preds), np.array(trues)
    return np.mean(preds == trues), preds, trues

train_accs, val_accs = [], []
best_val_acc, best_model, best_init = 0.0, None, None
print("Training for 20 epochs...")
for epoch in range(EPOCHS):
    tr_acc, tr_loss = train_epoch(model, optimizer)
    acc, _, _ = evaluate(model, val_loader)
    train_accs.append(tr_acc); val_accs.append(acc)
    if acc > best_val_acc:
        best_val_acc = acc
        best_model = deepcopy(model)
        best_init = {k: v.detach().clone() for k, v in model.state_dict().items()}  # LTH: keep init
    print(f"Epoch {epoch:02d} | Train Acc: {tr_acc:.4f} | Val Acc: {acc:.4f}")

plt.figure(figsize=(12, 6))
plt.plot(train_accs, label="Train Acc", color="navy")
plt.plot(val_accs, label="Val Acc", color="orange")
plt.title("Training & Validation Accuracy"); plt.xlabel("Epoch")
plt.ylabel("Accuracy"); plt.legend(); plt.grid(); plt.show()

## 7. Thermodynamic pruning threshold search (Lottery Ticket)


In [ ]:
PRUNE_THRESHOLDS = [0.01, 0.05, 0.10]
results = []

def prune_and_reset(model_copy, init_state, th):
    """Zero weights with |w| < th, reset survivors to initial values (lottery-ticket reset)."""
    kept, total = 0, 0
    for name, p in model_copy.named_parameters():
        if p.dim() > 1:
            mask = p.abs() >= th
            total += mask.numel(); kept += mask.sum().item()
            p.data = init_state[name].clone() * mask  # reset to init AND apply mask
    return kept / total

for th in PRUNE_THRESHOLDS:
    model_copy = SimpleMLP().to(DEVICE)                      # fresh init == lottery-ticket reset
    model_copy.load_state_dict(best_init)                    # start from ORIGINAL initialization
    keep_ratio = prune_and_reset(model_copy, best_init, th)  # apply magnitude mask
    print(f"\nPruning with th={th:.2f} | Kept {keep_ratio*100:.2f}% weights")

    opt = optim.AdamW(model_copy.parameters(), lr=1e-4, weight_decay=1e-3)
    acc_curve = []
    for epoch in range(FINETUNE_EPOCHS):
        train_epoch(model_copy, opt)
        acc, _, _ = evaluate(model_copy, val_loader)
        acc_curve.append(acc)
        print(f"Post-prune Epoch {epoch:02d}: Val Acc: {acc:.4f}")
    results.append((th, max(acc_curve), acc_curve))

best_th, best_prune_acc, _ = max(results, key=lambda x: x[1])
print(f"\nBEST THRESHOLD: {best_th:.2f} | Best Post-Prune Val Acc: {best_prune_acc:.4f}")

plt.figure(figsize=(10, 6))
for th, _, curve in results:
    plt.plot(curve, label=f"th={th}")
plt.title("Post-Prune Accuracy vs Epoch (Lottery Ticket Search)")
plt.xlabel("Fine-tune Epoch"); plt.ylabel("Val Accuracy")
plt.legend(); plt.grid(); plt.show()

## 8. Final model: prune with best threshold + fine-tune

In [ ]:
model_final = SimpleMLP().to(DEVICE)
model_final.load_state_dict(best_init)
keep_ratio = prune_and_reset(model_final, best_init, best_th)
print(f"Final model keeps {keep_ratio*100:.2f}% of weights")

optimizer = optim.AdamW(model_final.parameters(), lr=1e-4, weight_decay=1e-3)
for epoch in range(FINETUNE_EPOCHS):
    train_epoch(model_final, optimizer)
    acc, _, _ = evaluate(model_final, val_loader)
    print(f"Final Fine-tune Epoch {epoch:02d}: Val Acc: {acc:.4f}")

## 9. Evaluation on the held-out TEST set

In [ ]:
acc, preds, trues = evaluate(model_final, test_loader)
print(f"\nHeld-out TEST Accuracy: {acc:.4f}")
print("\nFINAL CLASSIFICATION REPORT")
print(classification_report(trues, preds, target_names=CLASS_NAMES))

cm = confusion_matrix(trues, preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("Confusion Matrix (Final Model, Test Set)")
plt.xlabel("Pred"); plt.ylabel("True")
plt.tight_layout(); plt.show()

# True sparsity of the final model
sparsity = sum((p.abs() < 1e-6).sum().item() for p in model_final.parameters() if p.dim() > 1)
total    = sum(p.numel() for p in model_final.parameters() if p.dim() > 1)
print(f"Final Sparsity: {sparsity/total*100:.2f}% ({sparsity:,} / {total:,} weights pruned)")

## Summary
| Stage | Val Acc |
|---|---|
| Dense MLP (20 ep) | ~0.74 |
| Prune th=0.01 + fine-tune | best |
| Prune th=0.05 + fine-tune | |
| Prune th=0.10 + fine-tune | |

**Improvements vs. the PDF:** leakage-free split, Kaiming init, real train-accuracy tracking, gradient clipping, lottery-ticket reset-to-init, held-out test set.